In [ ]:
""" OctoBench-data-filter.ipynb
This notebook deals with the OctoBench instances

Output Json File:

octo_instance_query.json      # Instructions from "UserQuery", "SystemPrompt", "CLAUDE.md" and "AGENTS.md" for each instance

"""
import re
import json

import pandas as pd
from datasets import load_dataset

from tqdm import tqdm
import json
from constants.config import init_env_and_logger

logger = init_env_and_logger()

# Load the OctoBench
OctoBench_id = "MiniMaxAI/OctoBench"
logger.info("Now Loading the origin OctoBench dataset")
ds = load_dataset(
    "json",
    data_files={
        "train": "hf://datasets/MiniMaxAI/OctoBench/OctoBench.jsonl"
    },
    split = "train"
)


In [ ]:
# Filter for multi-user_query

multi_query_ds = ds.filter(lambda x: len(x["user_query"]) > 1)
print(multi_query_ds)
print([len(x) for x in multi_query_ds["user_query"]])

In [ ]:
single_query_ds = ds.filter(lambda x: len(x["user_query"]) == 1)

# System_Prompt feature
sys = [x for x in ds["system_prompt"] if x != ""]
print(len(sys))

# Category Feature
category = set(ds["category"])
print(category)

sp_rows = ds.filter(lambda x: x["category"] == "SP")
print(sp_rows)

mem = ds.filter(lambda x: x["category"] == "memory")
print(mem)

skill = ds.filter(lambda x: x["category"] == "Skill")
print(skill)

claude = ds.filter(lambda x: x["category"] == "Claude.md")
print(claude)

agent = ds.filter(lambda x: x["category"] == "AGENTS.md")
print(agent)

UQ = ds.filter(lambda x: x["category"] == "User Query")
print(UQ)

print(claude[0]["workspace_abs_path"])

In [ ]:
import subprocess
import uuid
import os

def run(cmd):
    return subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def extract_from_image(image, path_pairs):
    container_name = f"tmp_{uuid.uuid4().hex[:8]}"
    image_exists = subprocess.run(
        ["docker", "image", "inspect", image],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    ).returncode == 0

    # Remove stale outputs so a missing source file cannot reuse an earlier incorrect extraction.
    for _, local_path in path_pairs:
        if os.path.exists(local_path):
            os.remove(local_path)

    try:
        # 1. Pull the image
        if not image_exists:
            run(["docker", "pull", "--platform=linux/amd64", image])

        # 2. Create the container
        run(["docker", "create", "--name", container_name, image])

        # 3. Copy File
        for container_path, local_path in path_pairs:
            try:
                run([
                    "docker", "cp",
                    f"{container_name}:{container_path}",
                    local_path
                ])
            except Exception as e:
                print(f"[ERROR] {image}_{container_name}_{container_path}: copy file error[{str(e)}]")
                continue

    except subprocess.CalledProcessError as e:
        print(f"[ERROR] {image}: {e.stderr.decode()}")
        return False

    finally:
        # 4. Delete Container
        subprocess.run(["docker", "rm", "-f", container_name],
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)


    return True

def docker_login(username, token):
        subprocess.run(
            ["docker", "login", "-u", username, "--password-stdin"],
            input=token.encode(),
            check=True
        )


def deal_one_instance(instance):
    """ 
    instance features:
    features: ['instance_id', 'user_query', 'system_prompt', 'category', 'image', 'workspace_abs_path', 'scaffold', 'checklist', 'expected_skill']

    Output: instance_query_dict:
    {
        "user_query": [xxx],
        "system_prompt": xxx,
        "Claude.md": xxx,
        "AGENT.md": xxx
        # Skill and Memory File is hard to extract, so not considered yet
        # Or maybe Skill can be downloaded ?
    }
    """

    qd = {
        "user_query" : instance["user_query"],
        "system_prompt" : instance["system_prompt"],
        "Claude.md" : "",
        "AGENTS.md" : ""
    }
    LOCAL_ADDR = f"../data/octo_image_file/{instance["instance_id"]}"
    os.makedirs(LOCAL_ADDR, exist_ok = True)

    CLAUDE_ADDR = LOCAL_ADDR + "/Claude.md"
    AGENTS_ADDR = LOCAL_ADDR + "/AGENTS.md"
    extracted = extract_from_image(
        instance["image"],
        [
            (instance["workspace_abs_path"] + "/CLAUDE.md", CLAUDE_ADDR),
            (instance["workspace_abs_path"] + "/AGENTS.md", AGENTS_ADDR),
        ],
    )
    if not extracted:
        return None

    try:
        with open(CLAUDE_ADDR, "r", encoding="utf-8") as f:
            claude_str = f.read()
            qd["Claude.md"] = claude_str
    except FileNotFoundError :
        logger.warning(f"Instance {instance["instance_id"]}: Claude.md not found")

    try:
        with open(AGENTS_ADDR, "r", encoding="utf-8") as f:
            agents_str = f.read()
            qd["AGENTS.md"] = agents_str
    except FileNotFoundError :
        logger.warning(f"Instance {instance["instance_id"]}: AGENTS.md not found")

    return qd


username = os.getenv("DOCKER_USERNAME")
token = os.getenv("DOCKER_TOKEN")
# print(username)
# print(token)

logger.info("Try login docker")
try:
    docker_login(username, token)
except Exception as e:
    logger.error(f"Docker login error:{str(e)}")

ds_without_mem_or_skill = ds.filter(lambda x: x["category"] not in ["memory", "Skill"])
instance_query_dict = {}
for instance in tqdm(ds_without_mem_or_skill):
    instance_query = deal_one_instance(instance)
    if instance_query is not None:
        instance_query_dict[instance["instance_id"]] = instance_query

from constants.addr import INSTANCE_QUERY_ADDR
with open(INSTANCE_QUERY_ADDR, "w", encoding="utf-8") as f:
    json.dump(instance_query_dict, f)


In [ ]:
from constants.addr import load_json, INSTANCE_QUERY_ADDR

instance_query_dict = load_json(INSTANCE_QUERY_ADDR)

Has_MD = {
    id: query
    for id,query in instance_query_dict.items()
    if query["Claude.md"] != "" or query["AGENTS.md"]!=""
}

print(json.dumps(Has_MD, indent = 4))
print(len(Has_MD))

In [ ]:
# Target Dataset: instance which mainly focuses on "User_query", "Claude.md", "AGENTS.md"

tds = ds.filter(lambda x: x["category"] in ["User Query", "Claude.md", "AGENTS.md"])

# print(json.dumps(tds.to_dict(), indent=4))
# print(tds.features)

# Filter the checklist by "UQ" and "Agents.md"
def filter_checklist(instance):
    instance["checklist"] = {
        category: category_dict
        for category, category_dict in instance["checklist"].items()
        if category in ["User query", "Agents.md"]
    }
    return instance

tds = tds.map(filter_checklist)
    # print(len(instance["checklist"]))
    # print(instance["instance_id"])
    # print(instance["checklist"].keys())
    # print(json.dumps(instance["checklist"], indent = 4))

# Test for 3 
tds = tds.select(range(3))

## CLAUDE.md is equal to AGENTS.md 
# for instance_id, qd in instance_query_dict.items():
#     print(instance_id, end="\t")
#     if qd["Claude.md"] != qd["AGENTS.md"]:
#         print("False")
#     else :
#         print("True")

In [ ]:
# Merge repository instructions into a realistic multi-turn UserQuery, then verify that the original checklist is reusable.

from copy import deepcopy
import os
import time
from typing import Any

from datasets import Dataset
from dotenv import load_dotenv
from litellm import OpenAI

load_dotenv(dotenv_path="./.env", override=True)
client = OpenAI(
    base_url=os.getenv("API_BASE_URL"),
    api_key=os.getenv("API_KEY"),
)
model = os.getenv("API_MODEL")


SPLIT_SYSTEM_PROMPT = """
You convert a coding benchmark request into a realistic sequence of user messages.
The messages simulate a user progressively supplying repository-specific instructions and tasks.
You must preserve the final coding task, its scope, constraints, and all observable acceptance criteria exactly.
Return JSON only; never solve the task or describe the transformation.
""".strip()

VERIFY_SYSTEM_PROMPT = """
You are a strict benchmark-data validator. Compare an original request with a proposed multi-turn rewrite.
Reject any rewrite that changes, weakens, strengthens, answers, or omits the task; invents requirements; or makes
any original checklist criterion no longer valid for grading the assistant's eventual workspace changes.
Return JSON only.
""".strip()

# Check The Query Type 
def _as_messages(user_query: Any) -> list[str]:
    if isinstance(user_query, str):
        return [user_query]
    if isinstance(user_query, list) and all(isinstance(item, str) for item in user_query):
        return user_query
    raise TypeError(f"user_query must be str or list[str], got {type(user_query)!r}")

# If the Claude.md is equal to AGENTS.md, preserve one.
def _merge_repository_instructions(query_source: dict[str, Any]) -> dict[str, str]:
    """Keep only one copy when CLAUDE.md and AGENTS.md are identical."""
    claude = (query_source.get("Claude.md") or "").strip()
    agents = (query_source.get("AGENTS.md") or "").strip()
    if not claude:
        return {"AGENTS.md": agents} if agents else {}
    if not agents:
        return {"Claude.md": claude}

    if claude == agents:
        return {"CLAUDE.md / AGENTS.md": claude}
    return {"Claude.md": claude, "AGENTS.md": agents}

# parse the llm-response-json
def _parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE)
    result = json.loads(text)
    if not isinstance(result, dict):
        raise ValueError("LLM response must be a JSON object")
    return result

# formulate the llm call
def _call_llm_json(
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0,
) -> dict[str, Any]:
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return _parse_json_object(response.choices[0].message.content)

# Split prompt
def build_split_prompt(
    instance: dict[str, Any],
    query_source: dict[str, Any],
    previous_issues: list[Any] | None = None,
) -> str:
    original_user_query = _as_messages(query_source.get("user_query", instance["user_query"]))
    payload = {
        "original_user_query": original_user_query,
        "repository_instructions": _merge_repository_instructions(query_source),
        "checklist_that_must_remain_valid": instance["checklist"],
    }
    retry_note = ""
    if previous_issues:
        retry_note = (
            "\nThe previous proposal was rejected for these reasons. Fix every issue:\n"
            + json.dumps(previous_issues, ensure_ascii=False)
        )

    return f"""
Create `user_query`, a chronological JSON array of one or more realistic user messages, from the input below.

Rules:
1. The cumulative meaning of all messages must contain the complete original final task without changing its
   requested behavior, scope, artifacts, constraints, or acceptance criteria. Preserve concrete identifiers, paths,
   commands, examples, and expected behavior needed by the task.
2. Turn relevant rules from Claude.md and AGENTS.md(in the "repository_instructions" section of the input) into natural user instructions and inject them before or with
   the subtask where they matter. Do not say they came from a file, system prompt, benchmark, or checklist.
3. You may decompose the task into multiple sequential subtasks, but do not add unrelated work, prerequisites,
   deliverables, or stricter requirements. Later turns may refine earlier turns but must not contradict them.
4. Do not perform the coding task, provide an answer, claim work is complete, or include assistant messages.
5. Cover every item in `checklist_that_must_remain_valid` without exception. Every checklist item must still grade
   the same final repository state after the assistant follows the entire message sequence. The checklist is evidence
   of required semantics, not text to quote.
6. Omit repository instructions that are wholly irrelevant to this task. Preserve every relevant instruction.
7. The final message must leave the assistant with an actionable task; the sequence must not end with context only.

Input JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}
{retry_note}

Output exactly in this format: {{"user_query": ["message 1", "message 2", ...]}}
""".strip()


def build_verification_prompt(
    instance: dict[str, Any],
    query_source: dict[str, Any],
    candidate_messages: list[str],
) -> str:
    checklist = instance["checklist"]
    payload = {
        "original_user_query": _as_messages(
            query_source.get("user_query", instance["user_query"])
        ),
        "repository_instructions": _merge_repository_instructions(query_source),
        "original_checklist": checklist,
        "candidate_user_query": candidate_messages,
    }
    return f"""
Validate the candidate as one cumulative user/assistant interaction context. Mark `valid` True only if:
- the original final task and all relevant repository instructions are preserved without semantic drift;
- splitting/reordering has introduced no contradictions, answers, or unrelated requirements;
- every item in `original_checklist` is covered without exception, remains supported by the candidate task, and can
  still grade the resulting workspace unchanged. Inspect every checklist item before deciding; checklist presence in
  the input alone does not count as coverage.

If your verification answer is "False", you should offer issues of the candidate in "issues" list.
If your verification answer is "True", leave the "issues" as blank list ([])

Input JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}

Output exactly in this Format:
{{
  "valid": True or False,
  "issues": [issue_text_1, issue_text_2, ...]
}}
""".strip()


def _validate_candidate_shape(candidate_messages: Any) -> list[str]:
    if not isinstance(candidate_messages, list) or not candidate_messages:
        return ["user_query must be a non-empty JSON array"]
    if not all(isinstance(message, str) and message.strip() for message in candidate_messages):
        return ["every user_query item must be a non-empty string"]
    return []


def split_and_verify_instance(
    instance: dict[str, Any],
    query_source: dict[str, Any],
    max_attempts: int = 3,
) -> list[str]:
    issues = []

    for attempt in range(1, max_attempts + 1):
        # Generate a candidate split user_query
        generated = _call_llm_json(
            SPLIT_SYSTEM_PROMPT,
            build_split_prompt(instance, query_source, previous_issues=issues),
            temperature=0.2,
        )

        candidate_messages = generated.get("user_query")

        # Probably Format Problems
        valid_shape = _validate_candidate_shape(candidate_messages) 
        if valid_shape:
            issues = [*issues, *valid_shape]
            continue

        # llm verification process
        verification = _call_llm_json(
            VERIFY_SYSTEM_PROMPT,
            build_verification_prompt(instance, query_source, candidate_messages),
            temperature=0,
        )

        # verified success
        if verification.get("valid") is True:
            return candidate_messages

        # Find issues in verification
        verificated_issues = verification.get("issues") or []
        if not isinstance(verificated_issues, list):
            verificated_issues = [str(verificated_issues)]
        
        issues = [*issues, *verificated_issues]

    raise ValueError(f"No checklist-preserving split after {max_attempts} attempts: {issues}")


# Only verified instances enter split_tds. The original checklist object is copied without modification.
SPLIT_QUERY_ADDR = "../data/octobench_split_user_query.jsonl"

if os.path.exists(SPLIT_QUERY_ADDR):
    existing_split_tds = Dataset.from_json(SPLIT_QUERY_ADDR)
    split_instances = existing_split_tds.to_list()
    logger.info(f"Loaded {len(split_instances)} verified instances from {SPLIT_QUERY_ADDR}")
else:
    split_instances = []

verified_instance_ids = {instance["instance_id"] for instance in split_instances}
print(tds)

for instance in tqdm(tds, desc="Splitting UserQuery"):
    instance_id = instance["instance_id"]
    if instance_id in verified_instance_ids:
        continue

    query_source = instance_query_dict.get(instance_id)

    # Has source
    if query_source is None:
        logger.warning(f"{instance_id} missing from instance_query_dict")
        continue

    try:
        split_user_query = split_and_verify_instance(instance, query_source)

        # Copy for a transformed instance
        transformed = deepcopy(dict(instance))
        transformed["original_user_query"] = _as_messages(
            query_source.get("user_query", instance["user_query"])
        )
        transformed["user_query"] = split_user_query
        transformed["checklist"] = deepcopy(instance["checklist"])
        split_instances.append(transformed)
        verified_instance_ids.add(instance_id)
    except Exception as error:
        logger.warning(f"Instance {instance_id} was not emitted: {error}")
    time.sleep(0.1)

split_tds = Dataset.from_list(split_instances)
logger.info(f"verified={len(split_instances)}, all={len(tds)}")

# from constants.addr import SPLIT_QUERY_ADDR
split_tds.to_json(SPLIT_QUERY_ADDR, force_ascii=False, indent=4)
